In [1]:
from dataclasses import dataclass
from typing import Dict

import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from typesafe_sdk import TypeSafeClient, Noul

load_dotenv()

True

In [2]:
problem = """
Uma empresa possui um sistema de atendimento ao cliente.

Atualmente, todos os tickets entram em uma única fila.
A empresa quer automatizar a triagem.

Proponha uma arquitetura que:

1. classifique os tickets por departamento;
2. identifique tickets urgentes;
3. identifique risco de cancelamento;
4. mantenha um humano no loop para casos incertos;
5. seja simples o suficiente para uma equipe pequena manter.
"""

In [3]:
JUDGE_CRITERIA = {
    "correctness": (
        "The proposed solution is logically correct and does not "
        "contain important technical contradictions."
    ),

    "completeness": (
        "The solution addresses all requirements stated in the problem."
    ),

    "feasibility": (
        "The proposed solution is realistically implementable by "
        "a small engineering team."
    ),

    "clarity": (
        "The solution is clear, structured, and sufficiently specific "
        "to be actionable."
    ),

    "simplicity": (
        "The architecture avoids unnecessary complexity and is appropriate "
        "for the stated constraints."
    ),
}

In [4]:
openai_client = OpenAI()

OPENAI_MODEL = "gpt-5-mini"

In [5]:
def ask_openai(
    problem: str,
    previous_solution: str | None = None,
    feedback: str | None = None,
) -> str:

    if previous_solution is None:
        prompt = f"""
Você é responsável por resolver o problema abaixo.

PROBLEMA:
{problem}

Produza uma solução clara, tecnicamente consistente e prática.

Não discuta o processo de avaliação.
Entregue somente a solução proposta.
"""
    else:
        prompt = f"""
Você está revisando uma solução anterior.

PROBLEMA:
{problem}

SOLUÇÃO ANTERIOR:
{previous_solution}

FEEDBACK DO AVALIADOR:
{feedback}

Produza uma nova versão da solução.

Regras:
- Corrija os problemas apontados pelo feedback.
- Preserve as partes da solução que já estavam boas.
- Não mencione o avaliador.
- Não explique o processo de revisão.
- Entregue apenas a solução final revisada.
"""

    response = openai_client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
    )

    return response.output_text

In [6]:
def evaluate_with_jev(
    problem: str,
    solution: str,
) -> Dict[str, float]:

    state = {
        "problem": problem,
        "proposed_solution": solution,
    }

    questions = {
        name: Noul(
            instructions=(
                f"Evaluate `proposed_solution` for the following criterion: "
                f"{description} "
                "Consider the requirements in `problem`."
            ),
            criteria={
                "true": (
                    "The proposed solution satisfies this criterion "
                    "to a strong degree."
                ),
                "false": (
                    "The proposed solution fails to satisfy this criterion "
                    "or has important deficiencies related to it."
                ),
            },
        )
        for name, description in JUDGE_CRITERIA.items()
    }

    with TypeSafeClient() as client:
        response = client.system_one(
            state=state,
            questions=questions,
        )

    return {
        name: response.answers[name].noul
        for name in questions
    }

In [7]:
def build_feedback(
    scores: Dict[str, float],
    threshold: float = 0.80,
) -> str:

    problems = []

    for criterion, score in scores.items():

        if score < threshold:

            description = JUDGE_CRITERIA[
                criterion
            ]

            problems.append(
                f"- Improve {criterion}: "
                f"{description} "
                f"Current confidence that the criterion is satisfied: "
                f"{score:.2f}."
            )

    if not problems:
        return (
            "All evaluation criteria are currently satisfied "
            "above the required threshold."
        )

    return (
        "The solution needs improvement in the following areas:\n"
        + "\n".join(problems)
    )

In [8]:
def passes_evaluation(
    scores: Dict[str, float],
    threshold: float = 0.80,
) -> bool:

    return all(
        score >= threshold
        for score in scores.values()
    )

In [9]:
@dataclass
class CollaborationRound:

    round_number: int

    solution: str

    scores: Dict[str, float]

    feedback: str

    passed: bool

In [10]:
def collaborate(
    problem: str,
    max_rounds: int = 5,
    threshold: float = 0.80,
):

    history = []

    solution = None
    feedback = None

    for round_number in range(
        1,
        max_rounds + 1,
    ):

        print(
            f"\n{'=' * 70}"
        )
        print(
            f"RODADA {round_number}"
        )
        print(
            f"{'=' * 70}"
        )

        # -----------------------------------
        # OpenAI propõe/revisa a solução
        # -----------------------------------

        solution = ask_openai(
            problem=problem,
            previous_solution=solution,
            feedback=feedback,
        )

        print("\nOPENAI:")
        print(solution)

        # -----------------------------------
        # Jev avalia
        # -----------------------------------

        scores = evaluate_with_jev(
            problem=problem,
            solution=solution,
        )

        print("\nJEV:")

        for criterion, score in scores.items():
            print(
                f"{criterion:15s}: "
                f"{score:.3f}"
            )

        # -----------------------------------
        # Política
        # -----------------------------------

        passed = passes_evaluation(
            scores,
            threshold=threshold,
        )

        feedback = build_feedback(
            scores,
            threshold=threshold,
        )

        print("\nFEEDBACK:")
        print(feedback)

        history.append(
            CollaborationRound(
                round_number=round_number,
                solution=solution,
                scores=scores,
                feedback=feedback,
                passed=passed,
            )
        )

        if passed:

            print(
                "\nCritérios atingidos. "
                "Colaboração encerrada."
            )

            break

    return solution, history

In [11]:
final_solution, history = collaborate(
    problem=problem,
    max_rounds=5,
    threshold=0.80,
)


RODADA 1

OPENAI:
Solução proposta — arquitetura prática, simples de manter e com humano no loop

Visão geral
Uma pipeline automática de triagem composta por: ingestão → normalização → inferência (classificador de departamento + detector de urgência + detector de risco de cancelamento) → roteador de decisão (automatiza ou encaminha para humano) → interface de atendimento com feedback. Usar modelos leves baseados em Transformer (ex.: DistilBERT/BERTimbau) ou classificadores simples com embeddings, combinados com regras simples para cobertura imediata. Projetar para operar com serviços gerenciados e componentes fáceis de manter (API REST, filas, banco relacional, ferramenta de rotulagem).

Componentes principais
1. Ingestão
- Fontes: e-mail, formulário web, chat, telefone -> centralizadas por um conector.
- Normalização: extrair texto, metadados (cliente, canal, horário, histórico).

2. Pré-processamento
- Limpeza e tokenização, normalização de linguagem (PT-BR), extração de features rá

In [12]:
rows = []

for item in history:

    row = {
        "round": item.round_number,
        "passed": item.passed,
    }

    row.update(
        item.scores
    )

    rows.append(row)


history_df = pd.DataFrame(rows)

history_df

,round,passed,correctness,completeness,feasibility,clarity,simplicity
0,1,False,0.93,0.97,0.80,0.95,0.73
1,2,True,0.92,0.97,0.87,0.95,0.88
